# N-Puzzle Solver using Depth-Limited Search (DLS)


In [1]:
%%writefile input.txt
Limit: 5
Start: 1 2 3 4 5 6 0 7 8
Goal: 1 2 3 4 5 6 7 8 0

Writing input.txt


In [2]:
import math
import sys

def get_grid_size(state):
    return int(math.sqrt(len(state)))

def get_moves(state):
    """Generates valid moves for the blank tile (0) in the current state."""
    moves = []
    grid_size = get_grid_size(state)
    blank_idx = state.index(0)
    row = blank_idx // grid_size
    col = blank_idx % grid_size

    # Directions: Up, Down, Left, Right
    if row > 0:
        moves.append(swap(state, blank_idx, blank_idx - grid_size)) # Up
    if row < grid_size - 1:
        moves.append(swap(state, blank_idx, blank_idx + grid_size)) # Down
    if col > 0:
        moves.append(swap(state, blank_idx, blank_idx - 1))         # Left
    if col < grid_size - 1:
        moves.append(swap(state, blank_idx, blank_idx + 1))         # Right

    return moves

def swap(state, i, j):
    """Swaps two elements in a tuple."""
    lst = list(state)
    lst[i], lst[j] = lst[j], lst[i]
    return tuple(lst)

def format_state(state):
    """Formats the tuple state into a grid string for display."""
    grid_size = get_grid_size(state)
    grid_str = ""
    for i in range(0, len(state), grid_size):
        row = state[i:i+grid_size]
        grid_str += " ".join(f"{str(x):>2}" for x in row) + "\n"
    return grid_str

def dls(current_state, goal_state, limit, path, path_set, output_file):
    """Recursive Depth-Limited Search function."""
    # Write intermediate state to output file
    output_file.write(f"Visiting Depth {len(path)-1}:\n{format_state(current_state)}\n")

    if current_state == goal_state:
        return path
    if limit == 0:
        return "cutoff"

    cutoff_occurred = False
    for next_state in get_moves(current_state):
        # Prevent cycles within the current path branch
        if next_state not in path_set:
            path.append(next_state)
            path_set.add(next_state)

            result = dls(next_state, goal_state, limit - 1, path, path_set, output_file)

            if result == "cutoff":
                cutoff_occurred = True
            elif result is not None:
                return result

            # Backtrack
            path.pop()
            path_set.remove(next_state)

    return "cutoff" if cutoff_occurred else None

def solve_puzzle(input_filename, output_filename):
    # Read inputs
    try:
        with open(input_filename, 'r') as f:
            lines = f.readlines()
            limit = int(lines[0].split(":")[1].strip())
            start_state = tuple(map(int, lines[1].split(":")[1].strip().split()))
            goal_state = tuple(map(int, lines[2].split(":")[1].strip().split()))
    except Exception as e:
        print(f"Error reading {input_filename}: {e}")
        return

    # Check for grid validity
    if int(math.sqrt(len(start_state)))**2 != len(start_state):
        print("Invalid state length. Must be a perfect square (e.g., 9 for 8-puzzle).")
        return

    print(f"Solving puzzle with limit: {limit}...\n")

    with open(output_filename, 'w') as out_f:
        out_f.write("--- DLS Intermediate Traversal ---\n\n")

        path = [start_state]
        path_set = {start_state}

        # Execute DLS
        result = dls(start_state, goal_state, limit, path, path_set, out_f)

        out_f.write("\n--- Final Result ---\n")
        if result == "cutoff":
            msg = f"Solution not found within depth limit of {limit}.\n"
            print(msg)
            out_f.write(msg)
        elif result is None:
            msg = "No solution exists.\n"
            print(msg)
            out_f.write(msg)
        else:
            msg = f"Solution found in {len(result) - 1} moves!\n"
            print(msg)
            out_f.write(msg)
            for i, state in enumerate(result):
                step_str = f"Step {i}:\n{format_state(state)}\n"
                print(step_str)
                out_f.write(step_str)

In [3]:
# Run the solver
solve_puzzle('input.txt', 'output.txt')

Solving puzzle with limit: 5...

Solution found in 2 moves!

Step 0:
 1  2  3
 4  5  6
 0  7  8


Step 1:
 1  2  3
 4  5  6
 7  0  8


Step 2:
 1  2  3
 4  5  6
 7  8  0




In [4]:
# Display the contents of the output file
with open('output.txt', 'r') as f:
    print(f.read())

--- DLS Intermediate Traversal ---

Visiting Depth 0:
 1  2  3
 4  5  6
 0  7  8

Visiting Depth 1:
 1  2  3
 0  5  6
 4  7  8

Visiting Depth 2:
 0  2  3
 1  5  6
 4  7  8

Visiting Depth 3:
 2  0  3
 1  5  6
 4  7  8

Visiting Depth 4:
 2  5  3
 1  0  6
 4  7  8

Visiting Depth 5:
 2  5  3
 1  7  6
 4  0  8

Visiting Depth 5:
 2  5  3
 0  1  6
 4  7  8

Visiting Depth 5:
 2  5  3
 1  6  0
 4  7  8

Visiting Depth 4:
 2  3  0
 1  5  6
 4  7  8

Visiting Depth 5:
 2  3  6
 1  5  0
 4  7  8

Visiting Depth 2:
 1  2  3
 5  0  6
 4  7  8

Visiting Depth 3:
 1  0  3
 5  2  6
 4  7  8

Visiting Depth 4:
 0  1  3
 5  2  6
 4  7  8

Visiting Depth 5:
 5  1  3
 0  2  6
 4  7  8

Visiting Depth 4:
 1  3  0
 5  2  6
 4  7  8

Visiting Depth 5:
 1  3  6
 5  2  0
 4  7  8

Visiting Depth 3:
 1  2  3
 5  7  6
 4  0  8

Visiting Depth 4:
 1  2  3
 5  7  6
 0  4  8

Visiting Depth 5:
 1  2  3
 0  7  6
 5  4  8

Visiting Depth 4:
 1  2  3
 5  7  6
 4  8  0

Visiting Depth 5:
 1  2  3
 5  7  0
 4  8  6